# EmpathAI Fine-tune (SFT + DPO)
Pipeline: Unsloth QLoRA 4-bit → SFT → DPO → Push HuggingFace

In [1]:
# 1. Install and verify runtime packages for NVIDIA L4 / CUDA 12.4.
# Use the active Jupyter kernel interpreter directly. This is stricter than !pip.
import importlib
import os
import platform
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

print("Kernel Python:", sys.executable)
print("Python version:", sys.version.split()[0])

if sys.version_info < (3, 9):
    raise RuntimeError("Unsloth requires Python >= 3.9. Please switch the notebook kernel.")

preloaded = [name for name in ("torch", "transformers", "trl", "unsloth", "peft") if name in sys.modules]
if preloaded:
    print("Warning: already imported before install:", preloaded)
    print("If versions below do not match, restart the kernel and run this cell first.")


def run(cmd, *, check=True):
    print("\n>>", " ".join(cmd))
    result = subprocess.run(cmd, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(cmd)}")
    return result


def pip_install(*args):
    return run([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--upgrade-strategy",
        "only-if-needed",
        *args,
    ])

# Keep pip tooling current enough for modern manylinux CUDA wheels.
run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "-U", "pip", "setuptools", "wheel"])

# First install the exact CUDA 12.4 PyTorch stack for L4.
pip_install(
    "torch==2.5.1",
    "torchvision==0.20.1",
    "torchaudio==2.5.1",
    "--index-url",
    "https://download.pytorch.org/whl/cu124",
)

importlib.invalidate_caches()
import torch

torch_base_version = torch.__version__.split("+")[0]
if torch_base_version != "2.5.1" or torch.version.cuda != "12.4":
    raise RuntimeError(
        f"Expected torch 2.5.1+cu124, got torch={torch.__version__}, cuda={torch.version.cuda}. "
        "Restart the kernel and run this install cell first."
    )

# Match Unsloth's published CUDA 12.4 + Torch 2.5.1 extra. This pins compatible xformers.
pip_install(
    "unsloth[cu124-torch251]==2026.6.1",
    "unsloth-zoo==2026.6.1",
    "trl==0.24.0",
    "torchao==0.15.0",
    "fastapi",
    "uvicorn",
    "pydantic",
    "nest-asyncio",
)

# pip check can report unrelated preinstalled notebook image conflicts, so keep it diagnostic.
pip_check = run([sys.executable, "-m", "pip", "check"], check=False)
if pip_check.returncode != 0:
    print("pip check reported conflicts. Critical imports below are still enforced.")

importlib.invalidate_caches()

# Patch torchao integer aliases before importing Unsloth.
for i in range(1, 8):
    if not hasattr(torch, f"int{i}"):
        setattr(torch, f"int{i}", getattr(torch, "int8"))
    if not hasattr(torch, f"uint{i}"):
        setattr(torch, f"uint{i}", getattr(torch, "uint8"))

# Import Unsloth before transformers/peft so its runtime patches apply correctly.
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset
from huggingface_hub import HfApi, create_repo, login, upload_folder
from torch.amp.grad_scaler import GradScaler
from transformers import TrainerCallback
from transformers.utils.notebook import NotebookProgressCallback
from trl import DPOConfig, DPOTrainer, SFTConfig, SFTTrainer

import accelerate
import bitsandbytes as bnb
import datasets
import huggingface_hub
import peft
import torchao
import transformers
import trl
from packaging.version import parse as parse_version

if platform.system().lower() == "linux":
    import triton
    import xformers
else:
    triton = None
    xformers = None

expected_versions = {
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "trl": trl.__version__,
    "datasets": datasets.__version__,
    "huggingface_hub": huggingface_hub.__version__,
    "accelerate": accelerate.__version__,
    "peft": peft.__version__,
    "bitsandbytes": getattr(bnb, "__version__", "unknown"),
    "torchao": getattr(torchao, "__version__", "unknown"),
}
if xformers is not None:
    expected_versions["xformers"] = getattr(xformers, "__version__", "unknown")
if triton is not None:
    expected_versions["triton"] = getattr(triton, "__version__", "unknown")
print("\nDependency versions:")
for key, value in expected_versions.items():
    print(f"  {key}: {value}")

if trl.__version__ != "0.24.0":
    raise RuntimeError(f"Expected trl==0.24.0, got {trl.__version__}")
if parse_version(transformers.__version__) > parse_version("5.5.0"):
    raise RuntimeError(f"Expected transformers<=5.5.0 for Unsloth 2026.6.1, got {transformers.__version__}")

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

if platform.system().lower() == "linux":
    cloudflared = Path("cloudflared")
    if not cloudflared.exists():
        urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            cloudflared,
        )
    cloudflared.chmod(0o755)
    print("cloudflared ready:", cloudflared.resolve())
else:
    print("Skipping cloudflared download on non-Linux platform.")

print("Dependencies OK. Continue with the next cell.")


Kernel Python: /opt/micromamba/envs/jupyterlab/bin/python3
Python version: 3.12.13

>> /opt/micromamba/envs/jupyterlab/bin/python3 -m pip install --no-cache-dir -U pip setuptools wheel

>> /opt/micromamba/envs/jupyterlab/bin/python3 -m pip install --no-cache-dir --upgrade-strategy only-if-needed torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.2/908.2 MB 360.2 MB/s  0:00:02a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 331.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 495.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 601.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 621.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 642.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 M

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wbi-euc-extension 1.0.2 requires pydantic~=1.10.0, but you have pydantic 2.13.4 which is incompatible.
scheduler-jupyter-plugin 0.1.6 requires pydantic~=1.10.0, but you have pydantic 2.13.4 which is incompatible.
dataproc-jupyter-plugin 0.1.96 requires pydantic~=1.10.0, but you have pydantic 2.13.4 which is incompatible.



>> /opt/micromamba/envs/jupyterlab/bin/python3 -m pip check
wbi-euc-extension 1.0.2 has requirement pydantic~=1.10.0, but you have pydantic 2.13.4.
scheduler-jupyter-plugin 0.1.6 has requirement pydantic~=1.10.0, but you have pydantic 2.13.4.
dataproc-jupyter-plugin 0.1.96 has requirement pydantic~=1.10.0, but you have pydantic 2.13.4.
pip check reported conflicts. Critical imports below are still enforced.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!

Dependency versions:
  torch: 2.5.1+cu124
  cuda: 12.4
  transformers: 5.5.0
  trl: 0.24.0
  datasets: 4.3.0
  huggingface_hub: 1.18.0
  accelerate: 1.13.0
  peft: 0.19.1
  bitsandbytes: 0.49.2
  torchao: 0.15.0
  xformers: 0.0.29.post1
  triton: 3.1.0
CUDA available: True
GPU: NVIDIA L4
BF16 supported: True
cloudflared r

In [2]:
# 1b. Patch torchao before importing Unsloth.
import torch

for i in range(1, 8):
    if not hasattr(torch, f"int{i}"):
        setattr(torch, f"int{i}", getattr(torch, "int8"))
    if not hasattr(torch, f"uint{i}"):
        setattr(torch, f"uint{i}", getattr(torch, "uint8"))

# Import Unsloth before transformers/peft so its patches are applied cleanly.
from unsloth import FastLanguageModel
from transformers import TrainerCallback

print("Unsloth import OK")


Unsloth import OK


In [3]:
# 1c. Runtime sanity check.
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this notebook.")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("BF16 supported:", torch.cuda.is_bf16_supported())
print("Free VRAM:", round(torch.cuda.mem_get_info()[0] / 1e9, 2), "GB")


GPU: NVIDIA L4
CUDA: 12.4
BF16 supported: True
Free VRAM: 23.46 GB


In [4]:
# 2. Login Hugging Face & configure local/GCS paths.
import os
import shutil
import subprocess
from pathlib import Path
from huggingface_hub import login
from transformers import TrainerCallback


def load_local_env(path=".env"):
    """Load simple KEY=VALUE pairs without printing secrets."""
    env_path = Path(path)
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


load_local_env()

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or ""

if HF_TOKEN.startswith("hf_"):
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face.")
else:
    print("HF_TOKEN is not set. Public datasets can still load; push_to_hub requires a write token.")

RUN_NAME = os.environ.get("EMPATHAI_RUN_NAME", "l4-v2")

GCS_BUCKET = os.environ.get("GCS_BUCKET", "empathai-llama")
GCS_BASE_DIR = f"gs://{GCS_BUCKET}/empathAI-finetune"

SFT_OUTPUT_DIR = f"empathAI-sft-{RUN_NAME}"
DPO_OUTPUT_DIR = f"empathAI-dpo-{RUN_NAME}"
LOCAL_ADAPTER_DIR = f"empathAI-lora-adapter-{RUN_NAME}"

GCS_SFT_DIR = f"{GCS_BASE_DIR}/sft-{RUN_NAME}"
GCS_DPO_DIR = f"{GCS_BASE_DIR}/dpo-{RUN_NAME}"
GCS_ADAPTER_DIR = f"{GCS_BASE_DIR}/adapter-{RUN_NAME}"

print(f"Run: {RUN_NAME}")
print(f"SFT: {SFT_OUTPUT_DIR} -> {GCS_SFT_DIR}")
print(f"DPO: {DPO_OUTPUT_DIR} -> {GCS_DPO_DIR}")
print(f"Adapter: {LOCAL_ADAPTER_DIR} -> {GCS_ADAPTER_DIR}")

GSUTIL_AVAILABLE = shutil.which("gsutil") is not None
ENABLE_GCS_SYNC = os.environ.get("ENABLE_GCS_SYNC", "0") == "1"

if GSUTIL_AVAILABLE:
    print("gsutil found.")
else:
    print("WARNING: gsutil not found. GCS sync/restore will be skipped.")
if not ENABLE_GCS_SYNC:
    print("GCS sync/restore disabled by default. Set ENABLE_GCS_SYNC=1 to enable checkpoint sync.")


class GCSSyncCallback(TrainerCallback):
    """Sync checkpoints to GCS after Trainer saves when explicitly enabled."""

    def __init__(self, local_dir, gcs_dir):
        self.local_dir = local_dir
        self.gcs_dir = gcs_dir

    def on_save(self, args, state, control, **kwargs):
        if not ENABLE_GCS_SYNC:
            return control
        if not GSUTIL_AVAILABLE:
            print("Skipping GCS sync because gsutil is not installed.")
            return control
        if not os.path.exists(self.local_dir):
            print(f"Skipping GCS sync: local dir not found: {self.local_dir}")
            return control

        print(f"Syncing {self.local_dir} -> {self.gcs_dir} ...", flush=True)
        ret = subprocess.run(
            ["gsutil", "-m", "rsync", "-r", self.local_dir, self.gcs_dir],
            capture_output=True,
            text=True,
            timeout=600,
        )

        if ret.returncode == 0:
            print("GCS sync OK", flush=True)
        else:
            print("GCS sync failed:", flush=True)
            print(ret.stderr[:1000], flush=True)
        return control


def restore_from_gcs(local_dir, gcs_dir):
    """Restore checkpoints from GCS when enabled and local output dir is empty."""
    os.makedirs(local_dir, exist_ok=True)

    if any(d.startswith("checkpoint-") for d in os.listdir(local_dir)):
        print(f"Local checkpoints already exist in {local_dir}; skipping GCS restore.")
        return

    if not ENABLE_GCS_SYNC:
        print("Skipping GCS restore. Set ENABLE_GCS_SYNC=1 to restore checkpoints from GCS.")
        return
    if not GSUTIL_AVAILABLE:
        print("Skipping GCS restore because gsutil is not installed.")
        return

    print(f"Restoring checkpoints from {gcs_dir} -> {local_dir} ...", flush=True)
    ret = subprocess.run(
        ["gsutil", "-m", "rsync", "-r", gcs_dir, local_dir],
        capture_output=True,
        text=True,
        timeout=600,
    )

    if ret.returncode != 0:
        print("GCS restore failed or folder does not exist yet:", flush=True)
        print(ret.stderr[:1000], flush=True)

    ckpts = [
        d for d in os.listdir(local_dir)
        if d.startswith("checkpoint-") and os.path.isdir(os.path.join(local_dir, d))
    ]
    print(f"Restored {len(ckpts)} checkpoint(s)." if ckpts else "No GCS checkpoint found; training from scratch.")


def get_resume_checkpoint(local_dir):
    """Return the newest checkpoint path, or None."""
    if not os.path.isdir(local_dir):
        return None

    ckpts = [
        d for d in os.listdir(local_dir)
        if d.startswith("checkpoint-") and os.path.isdir(os.path.join(local_dir, d))
    ]
    if not ckpts:
        return None

    latest = max(ckpts, key=lambda x: int(x.split("-")[-1]))
    return os.path.join(local_dir, latest)


print("Helper functions loaded.")


Logged in to Hugging Face.
Run: l4-v2
SFT: empathAI-sft-l4-v2 -> gs://empathai-llama/empathAI-finetune/sft-l4-v2
DPO: empathAI-dpo-l4-v2 -> gs://empathai-llama/empathAI-finetune/dpo-l4-v2
Adapter: empathAI-lora-adapter-l4-v2 -> gs://empathai-llama/empathAI-finetune/adapter-l4-v2
gsutil found.
GCS sync/restore disabled by default. Set ENABLE_GCS_SYNC=1 to enable checkpoint sync.
Helper functions loaded.


In [5]:
# 3. Load model and attach LoRA adapters.
from unsloth import FastLanguageModel
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("Free VRAM:", round(torch.cuda.mem_get_info()[0] / 1e9, 2), "GB")

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    token=HF_TOKEN or None,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

model.print_trainable_parameters()


GPU: NVIDIA L4
Free VRAM: 23.46 GB
==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.6.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 83,886,080 || all params: 8,114,147,328 || trainable%: 1.0338


In [6]:
# 4. Load SFT dataset as conversational prompt-completion rows.
from datasets import load_dataset

dataset_id = "thanhhoangnvbg/empathAI-dpo-vi"
print(f"Loading dataset from: {dataset_id}")

sft_train = load_dataset(dataset_id, data_files="sft_train.jsonl", split="train")
sft_dev   = load_dataset(dataset_id, data_files="sft_dev.jsonl",   split="train")


def clean_chat_messages(msgs):
    """Normalize message lists to plain {role, content} dicts."""
    if not isinstance(msgs, list):
        return []
    result = []
    for msg in msgs:
        role = msg.get("role") if hasattr(msg, "get") else None
        content = msg.get("content") if hasattr(msg, "get") else None
        if role is not None and content is not None:
            result.append({"role": str(role), "content": str(content)})
    return result


def normalize_sft(example):
    messages = clean_chat_messages(example["messages"])
    if len(messages) < 2 or messages[-1]["role"] != "assistant":
        return {"prompt": None, "completion": None}

    prompt = messages[:-1]
    completion = [messages[-1]]
    if not prompt:
        return {"prompt": None, "completion": None}

    return {"prompt": prompt, "completion": completion}


sft_train = sft_train.map(normalize_sft, batched=False, remove_columns=sft_train.column_names)
sft_dev   = sft_dev.map(normalize_sft,   batched=False, remove_columns=sft_dev.column_names)

before_train = len(sft_train)
before_dev = len(sft_dev)
sft_train = sft_train.filter(lambda x: x["prompt"] is not None)
sft_dev   = sft_dev.filter(lambda x: x["prompt"] is not None)

print(f"SFT train: {len(sft_train)} rows (dropped {before_train - len(sft_train)}) | dev: {len(sft_dev)} rows (dropped {before_dev - len(sft_dev)})")
print("SFT sample roles:", [m["role"] for m in sft_train[0]["prompt"]], "->", sft_train[0]["completion"][-1]["role"])


Loading dataset from: thanhhoangnvbg/empathAI-dpo-vi


README.md:   0%|          | 0.00/3.62k [00:00<?, ?B/s]

sft_train.jsonl:   0%|          | 0.00/13.3M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

sft_dev.jsonl:   0%|          | 0.00/1.68M [00:00<?, ?B/s]

...

Map:   0%|          | 0/7982 [00:00<?, ? examples/s]

Map:   0%|          | 0/1016 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7982 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1016 [00:00<?, ? examples/s]

SFT train: 7982 rows (dropped 0) | dev: 1016 rows (dropped 0)
SFT sample roles: ['system', 'user'] -> assistant


In [7]:
# 5. Train SFT.
import inspect
import os
import time
import torch
import datasets as _ds
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from transformers import TrainerCallback
from transformers.utils.notebook import NotebookProgressCallback

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")

SFT_MAX_LENGTH = int(os.environ.get("SFT_MAX_LENGTH", "1024"))
SFT_SAVE_STEPS = int(os.environ.get("SFT_SAVE_STEPS", "50"))
SFT_EVAL_STEPS = int(os.environ.get("SFT_EVAL_STEPS", str(SFT_SAVE_STEPS)))
SFT_LOG_STEPS = int(os.environ.get("SFT_LOG_STEPS", "1"))
print(f"SFT max_length={SFT_MAX_LENGTH}, eval_steps={SFT_EVAL_STEPS}, save_steps={SFT_SAVE_STEPS}, logging_steps={SFT_LOG_STEPS}")

sft_sig = inspect.signature(SFTTrainer.__init__)
trainer_kwargs = {}
if "processing_class" in sft_sig.parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

sft_config_sig = inspect.signature(SFTConfig.__init__)
sft_config_kwargs = dict(
    dataset_num_proc=1,
    packing=True,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=30,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_strategy="steps",
    logging_first_step=True,
    logging_steps=SFT_LOG_STEPS,
    eval_strategy="steps",
    eval_steps=SFT_EVAL_STEPS,
    save_strategy="steps",
    save_steps=SFT_SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=0.3,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    dataloader_persistent_workers=False,
    disable_tqdm=True,
    seed=3407,
    output_dir=SFT_OUTPUT_DIR,
    report_to="none",
)
if "max_length" in sft_config_sig.parameters:
    sft_config_kwargs["max_length"] = SFT_MAX_LENGTH
else:
    sft_config_kwargs["max_seq_length"] = SFT_MAX_LENGTH
if "completion_only_loss" in sft_config_sig.parameters:
    sft_config_kwargs["completion_only_loss"] = True

_orig_ds_map = _ds.Dataset.map
def _safe_map(self, *args, **kwargs):
    kwargs["num_proc"] = 1
    return _orig_ds_map(self, *args, **kwargs)
_ds.Dataset.map = _safe_map

try:
    sft_trainer = SFTTrainer(
        model=model,
        **trainer_kwargs,
        train_dataset=sft_train,
        eval_dataset=sft_dev,
        args=SFTConfig(**sft_config_kwargs),
    )
finally:
    _ds.Dataset.map = _orig_ds_map

try:
    sft_trainer.remove_callback(NotebookProgressCallback)
except ValueError:
    pass

_orig_sft_compute_loss = sft_trainer.compute_loss
def _sft_patched_compute_loss(*args, **kwargs):
    kwargs.pop("num_items_in_batch", None)
    return _orig_sft_compute_loss(*args, **kwargs)
sft_trainer.compute_loss = _sft_patched_compute_loss

class HeartbeatCallback(TrainerCallback):
    def __init__(self):
        self.started = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        step = int(state.global_step)
        if step <= 10 or step % 10 == 0:
            elapsed = time.time() - self.started
            alloc = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
            reserved = torch.cuda.memory_reserved() / 1e9 if torch.cuda.is_available() else 0.0
            print(
                f"[heartbeat] step={step}/{state.max_steps} epoch={state.epoch:.4f} "
                f"elapsed={elapsed/60:.1f}m vram_alloc={alloc:.2f}GB vram_reserved={reserved:.2f}GB",
                flush=True,
            )
        return control

sft_trainer.add_callback(HeartbeatCallback())
sft_trainer.add_callback(GCSSyncCallback(SFT_OUTPUT_DIR, GCS_SFT_DIR))
restore_from_gcs(SFT_OUTPUT_DIR, GCS_SFT_DIR)
_sft_resume = get_resume_checkpoint(SFT_OUTPUT_DIR)

print("Starting SFT training...", flush=True)
print(f"Resume checkpoint: {_sft_resume}" if _sft_resume else "Training from scratch", flush=True)
sft_trainer.train(resume_from_checkpoint=_sft_resume)
print("SFT training complete", flush=True)


SFT max_length=1024, eval_steps=50, save_steps=50, logging_steps=1


Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=1):   0%|          | 0/7982 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=1):   0%|          | 0/7982 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=1):   0%|          | 0/1016 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=1):   0%|          | 0/1016 [00:00<?, ? examples/s]

...

{'loss': '2.866', 'grad_norm': '3.236', 'learning_rate': '1.418e-05', 'epoch': '2.52'}
{'loss': '2.51', 'grad_norm': '3.19', 'learning_rate': '1.387e-05', 'epoch': '2.526'}
{'loss': '2.294', 'grad_norm': '2.985', 'learning_rate': '1.355e-05', 'epoch': '2.531'}
{'loss': '3.31', 'grad_norm': '3.568', 'learning_rate': '1.324e-05', 'epoch': '2.537'}
{'loss': '2.434', 'grad_norm': '3.045', 'learning_rate': '1.294e-05', 'epoch': '2.543'}
{'loss': '2.126', 'grad_norm': '2.729', 'learning_rate': '1.263e-05', 'epoch': '2.548'}
{'loss': '2.575', 'grad_norm': '3.048', 'learning_rate': '1.233e-05', 'epoch': '2.554'}
{'loss': '2.21', 'grad_norm': '3.019', 'learning_rate': '1.204e-05', 'epoch': '2.559'}
{'loss': '1.859', 'grad_norm': '2.802', 'learning_rate': '1.174e-05', 'epoch': '2.565'}
[heartbeat] step=460/537 epoch=2.5706 elapsed=162.6m vram_alloc=6.53GB vram_reserved=8.23GB
{'loss': '3.027', 'grad_norm': '3.997', 'learning_rate': '1.145e-05', 'epoch': '2.571'}
{'loss': '1.835', 'grad_norm': '2

Unsloth: Restored added_tokens_decoder metadata in empathAI-sft-l4-v2/checkpoint-500/tokenizer_config.json.


{'loss': '2.609', 'grad_norm': '3.28', 'learning_rate': '2.617e-06', 'epoch': '2.8'}
{'loss': '2.262', 'grad_norm': '3.068', 'learning_rate': '2.478e-06', 'epoch': '2.806'}
{'loss': '1.958', 'grad_norm': '2.879', 'learning_rate': '2.343e-06', 'epoch': '2.811'}
{'loss': '2.742', 'grad_norm': '3.309', 'learning_rate': '2.211e-06', 'epoch': '2.817'}
{'loss': '2.123', 'grad_norm': '3.22', 'learning_rate': '2.083e-06', 'epoch': '2.822'}
{'loss': '1.834', 'grad_norm': '2.896', 'learning_rate': '1.959e-06', 'epoch': '2.828'}
{'loss': '2.737', 'grad_norm': '3.514', 'learning_rate': '1.839e-06', 'epoch': '2.834'}
{'loss': '2.168', 'grad_norm': '2.888', 'learning_rate': '1.723e-06', 'epoch': '2.839'}
{'loss': '2.974', 'grad_norm': '3.214', 'learning_rate': '1.61e-06', 'epoch': '2.845'}
[heartbeat] step=510/537 epoch=2.8503 elapsed=180.3m vram_alloc=6.53GB vram_reserved=8.22GB
{'loss': '2.831', 'grad_norm': '3.112', 'learning_rate': '1.501e-06', 'epoch': '2.85'}
{'loss': '1.735', 'grad_norm': '2.

Unsloth: Restored added_tokens_decoder metadata in empathAI-sft-l4-v2/checkpoint-537/tokenizer_config.json.


{'train_runtime': '1.145e+04', 'train_samples_per_second': '0.375', 'train_steps_per_second': '0.047', 'train_loss': '4.044', 'epoch': '3'}
SFT training complete


In [9]:
# 5b. Evaluate SFT on test set with completion-only labels.
from transformers.utils.notebook import NotebookProgressCallback

sft_test = load_dataset(dataset_id, data_files="sft_test.jsonl", split="train")
sft_test = sft_test.map(normalize_sft, batched=False, remove_columns=sft_test.column_names)
sft_test = sft_test.filter(lambda x: x["prompt"] is not None)
print(f"SFT test: {len(sft_test)} rows")


def tokenize_sft_prompt_completion(example, max_length=2048):
    prompt_text = tokenizer.apply_chat_template(
        example["prompt"], tokenize=False, add_generation_prompt=True
    )
    full_text = tokenizer.apply_chat_template(
        example["prompt"] + example["completion"], tokenize=False, add_generation_prompt=False
    )

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]

    if len(full_ids) > max_length:
        overflow = len(full_ids) - max_length
        full_ids = full_ids[overflow:]
        prompt_label_len = max(len(prompt_ids) - overflow, 0)
    else:
        prompt_label_len = len(prompt_ids)

    labels = [-100] * min(prompt_label_len, len(full_ids)) + full_ids[min(prompt_label_len, len(full_ids)):]
    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }


sft_test_tok = sft_test.map(tokenize_sft_prompt_completion, remove_columns=sft_test.column_names)
sft_trainer.remove_callback(NotebookProgressCallback)

sft_test_results = sft_trainer.evaluate(eval_dataset=sft_test_tok)
print("SFT Test Metrics:", {k: round(v, 4) for k, v in sft_test_results.items() if isinstance(v, float)})
print(f"SFT Test Loss: {sft_test_results.get('eval_loss', 'N/A')}")


sft_test.jsonl:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1002 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1002 [00:00<?, ? examples/s]

SFT test: 1002 rows


Map:   0%|          | 0/1002 [00:00<?, ? examples/s]

{'eval_loss': '0.412', 'eval_runtime': '236.8', 'eval_samples_per_second': '4.232', 'eval_steps_per_second': '4.232', 'epoch': '3'}
SFT Test Metrics: {'eval_loss': 0.412, 'eval_runtime': 236.759, 'eval_samples_per_second': 4.232, 'eval_steps_per_second': 4.232, 'epoch': 3.0}
SFT Test Loss: 0.4120098948478699


In [10]:
# 6. Load DPO dataset.
dpo_train = load_dataset(dataset_id, data_files="dpo_train.jsonl", split="train")
dpo_dev   = load_dataset(dataset_id, data_files="dpo_dev.jsonl",   split="train")

sample = dpo_train[0]
print("Keys:", list(sample.keys()))
print("Prompt type:", type(sample["prompt"]))
print("Prompt[0] type:", type(sample["prompt"][0]))
print("Prompt[0]:", sample["prompt"][0])


def clean_msgs(msgs):
    """Normalize message lists to plain {role, content} dicts."""
    if not isinstance(msgs, list):
        msgs = [msgs]
    result = []
    for msg in msgs:
        role = msg.get("role") if hasattr(msg, "get") else None
        content = msg.get("content") if hasattr(msg, "get") else None
        if role is not None and content is not None:
            result.append({"role": str(role), "content": str(content)})
    return result


def normalize_dpo(example):
    prompt = clean_msgs(example["prompt"])
    chosen = clean_msgs(example["chosen"])
    rejected = clean_msgs(example["rejected"])

    if not prompt or not chosen or not rejected:
        return {"prompt": None, "chosen": None, "rejected": None}
    if chosen[-1]["role"] != "assistant" or rejected[-1]["role"] != "assistant":
        return {"prompt": None, "chosen": None, "rejected": None}

    # Keep conversational DPO format. DPOTrainer will apply the chat template exactly once.
    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}


dpo_train = dpo_train.map(normalize_dpo)
dpo_dev   = dpo_dev.map(normalize_dpo)

before_train = len(dpo_train)
before_dev = len(dpo_dev)
dpo_train = dpo_train.filter(lambda x: x["prompt"] is not None)
dpo_dev   = dpo_dev.filter(lambda x: x["prompt"] is not None)

print(f"DPO train: {len(dpo_train)} rows (dropped {before_train - len(dpo_train)}) | dev: {len(dpo_dev)} rows (dropped {before_dev - len(dpo_dev)})")
print("DPO sample roles:", [m["role"] for m in dpo_train[0]["prompt"]], "->", dpo_train[0]["chosen"][-1]["role"])


dpo_train.jsonl:   0%|          | 0.00/10.1M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

dpo_dev.jsonl:   0%|          | 0.00/1.27M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Keys: ['prompt', 'chosen', 'rejected']
Prompt type: <class 'list'>
Prompt[0] type: <class 'dict'>
Prompt[0]: {'role': 'system', 'content': 'Bạn là EmpathAI, trợ lý chăm sóc khách hàng e-commerce tiếng Việt, chuyên xử lý khách hàng đang bực tức hoặc phàn nàn.\n\nHãy đồng cảm vừa đủ, bình tĩnh, không đôi co. Không tự bịa chính sách, trạng thái đơn hàng, thời gian hoàn tiền, kết quả xử lý hoặc mức bồi thường. Nếu có chính sách/ngữ cảnh được cung cấp, hãy bám vào đó. Nếu thiếu thông tin, hãy hỏi thêm hoặc đề xuất chuyển bộ phận phụ trách kiểm tra. Nếu khách cung cấp mã đơn, xem đó là thông tin để kiểm tra, không tự kết luận trạng thái đơn nếu chưa có kết quả tra cứu. Không tiết lộ hoặc xác nhận thông tin cá nhân của người khác. Luôn đưa bước tiếp theo rõ ràng.'}


Map:   0%|          | 0/5139 [00:00<?, ? examples/s]

Map:   0%|          | 0/651 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5139 [00:00<?, ? examples/s]

Filter:   0%|          | 0/651 [00:00<?, ? examples/s]

DPO train: 5139 rows (dropped 0) | dev: 651 rows (dropped 0)
DPO sample roles: ['system', 'user', 'assistant', 'user'] -> assistant


In [11]:
# 7. Train DPO.
import inspect
import os
import time
import torch
from torch.amp.grad_scaler import GradScaler
import datasets as _ds
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported
from transformers import TrainerCallback
from transformers.utils.notebook import NotebookProgressCallback

DPO_MAX_LENGTH = int(os.environ.get("DPO_MAX_LENGTH", "1536"))
DPO_MAX_PROMPT_LENGTH = int(os.environ.get("DPO_MAX_PROMPT_LENGTH", "1024"))
DPO_SAVE_STEPS = int(os.environ.get("DPO_SAVE_STEPS", "50"))
DPO_EVAL_STEPS = int(os.environ.get("DPO_EVAL_STEPS", str(DPO_SAVE_STEPS)))
DPO_LOG_STEPS = int(os.environ.get("DPO_LOG_STEPS", "1"))
print(f"DPO max_length={DPO_MAX_LENGTH}, max_prompt_length={DPO_MAX_PROMPT_LENGTH}, eval_steps={DPO_EVAL_STEPS}, save_steps={DPO_SAVE_STEPS}")

dpo_sig = inspect.signature(DPOTrainer.__init__)
dpo_kwargs = {}
if "processing_class" in dpo_sig.parameters:
    dpo_kwargs["processing_class"] = tokenizer
else:
    dpo_kwargs["tokenizer"] = tokenizer

_orig_ds_map = _ds.Dataset.map
def _safe_map(self, *args, **kwargs):
    kwargs["num_proc"] = 1
    return _orig_ds_map(self, *args, **kwargs)
_ds.Dataset.map = _safe_map

try:
    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None,
        train_dataset=dpo_train,
        eval_dataset=dpo_dev,
        **dpo_kwargs,
        args=DPOConfig(
            beta=0.1,
            max_length=DPO_MAX_LENGTH,
            max_prompt_length=DPO_MAX_PROMPT_LENGTH,
            truncation_mode="keep_end",
            num_train_epochs=1,
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=8,
            learning_rate=3e-5,
            lr_scheduler_type="cosine",
            warmup_ratio=0.03,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            logging_strategy="steps",
            logging_first_step=True,
            logging_steps=DPO_LOG_STEPS,
            eval_strategy="steps",
            eval_steps=DPO_EVAL_STEPS,
            save_strategy="steps",
            save_steps=DPO_SAVE_STEPS,
            save_total_limit=3,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            optim="adamw_8bit",
            max_grad_norm=0.3,
            gradient_checkpointing=True,
            remove_unused_columns=False,
            dataloader_num_workers=0,
            dataloader_pin_memory=False,
            dataloader_persistent_workers=False,
            disable_tqdm=True,
            seed=3407,
            output_dir=DPO_OUTPUT_DIR,
            report_to="none",
        ),
    )
finally:
    _ds.Dataset.map = _orig_ds_map

try:
    dpo_trainer.remove_callback(NotebookProgressCallback)
except ValueError:
    pass

_orig_dpo_compute_loss = dpo_trainer.compute_loss
def _dpo_patched_compute_loss(*args, **kwargs):
    kwargs.pop("num_items_in_batch", None)
    return _orig_dpo_compute_loss(*args, **kwargs)
dpo_trainer.compute_loss = _dpo_patched_compute_loss

_orig_dpo_log = dpo_trainer.log
def _dpo_patched_log(logs, *args, **kwargs):
    return _orig_dpo_log(logs)
dpo_trainer.log = _dpo_patched_log

if not is_bfloat16_supported() and not hasattr(GradScaler, "_orig_unscale_grads"):
    GradScaler._orig_unscale_grads = GradScaler._unscale_grads_
    def _patched_unscale_grads(self, optimizer, inv_scale, found_inf, allow_fp16):
        return GradScaler._orig_unscale_grads(self, optimizer, inv_scale, found_inf, True)
    GradScaler._unscale_grads_ = _patched_unscale_grads

class DPOHeartbeatCallback(TrainerCallback):
    def __init__(self):
        self.started = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        step = int(state.global_step)
        if step <= 10 or step % 10 == 0:
            elapsed = time.time() - self.started
            alloc = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
            reserved = torch.cuda.memory_reserved() / 1e9 if torch.cuda.is_available() else 0.0
            print(
                f"[dpo-heartbeat] step={step}/{state.max_steps} epoch={state.epoch:.4f} "
                f"elapsed={elapsed/60:.1f}m vram_alloc={alloc:.2f}GB vram_reserved={reserved:.2f}GB",
                flush=True,
            )
        return control

dpo_trainer.add_callback(DPOHeartbeatCallback())
dpo_trainer.add_callback(GCSSyncCallback(DPO_OUTPUT_DIR, GCS_DPO_DIR))
restore_from_gcs(DPO_OUTPUT_DIR, GCS_DPO_DIR)
_dpo_resume = get_resume_checkpoint(DPO_OUTPUT_DIR)

print("Starting DPO training...", flush=True)
print(f"Resume checkpoint: {_dpo_resume}" if _dpo_resume else "Training from scratch", flush=True)
dpo_trainer.train(resume_from_checkpoint=_dpo_resume)
print("DPO training complete", flush=True)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


DPO max_length=1536, max_prompt_length=1024, eval_steps=50, save_steps=50


Extracting prompt in train dataset (num_proc=1):   0%|          | 0/5139 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=1):   0%|          | 0/5139 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=1):   0%|          | 0/5139 [00:00<?, ? examples/s]

...

{'loss': '1.976e-06', 'grad_norm': '0.0001044', 'learning_rate': '3.513e-07', 'rewards/chosen': '12.32', 'rewards/rejected': '-6.019', 'rewards/accuracies': '1', 'rewards/margins': '18.34', 'logps/chosen': '-44', 'logps/rejected': '-192.7', 'logits/chosen': '0.6154', 'logits/rejected': '0.3058', 'epoch': '0.9356'}
{'loss': '1.631e-06', 'grad_norm': '7.343e-05', 'learning_rate': '3.352e-07', 'rewards/chosen': '13.24', 'rewards/rejected': '-6.941', 'rewards/accuracies': '1', 'rewards/margins': '20.18', 'logps/chosen': '-29.28', 'logps/rejected': '-187.8', 'logits/chosen': '0.7994', 'logits/rejected': '0.2595', 'epoch': '0.9371'}
{'loss': '5.773e-08', 'grad_norm': '4.351e-06', 'learning_rate': '3.195e-07', 'rewards/chosen': '14.83', 'rewards/rejected': '-8.331', 'rewards/accuracies': '1', 'rewards/margins': '23.16', 'logps/chosen': '-29', 'logps/rejected': '-252.7', 'logits/chosen': '0.7912', 'logits/rejected': '0.2756', 'epoch': '0.9387'}
{'loss': '7.05e-06', 'grad_norm': '0.0004463', 'l

/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and

{'eval_loss': '0.002075', 'eval_runtime': '466.4', 'eval_samples_per_second': '1.396', 'eval_steps_per_second': '1.396', 'eval_rewards/chosen': '11.14', 'eval_rewards/rejected': '-6.379', 'eval_rewards/accuracies': '1', 'eval_rewards/margins': '17.52', 'eval_logps/chosen': '-53.43', 'eval_logps/rejected': '-194.2', 'eval_logits/chosen': '0.7284', 'eval_logits/rejected': '0.312', 'epoch': '1'}


Unsloth: Restored added_tokens_decoder metadata in empathAI-dpo-l4-v2/checkpoint-643/tokenizer_config.json.


{'train_runtime': '1.328e+04', 'train_samples_per_second': '0.387', 'train_steps_per_second': '0.048', 'train_loss': '0.0002717', 'epoch': '1'}
DPO training complete


In [12]:
# 7b. Evaluate DPO on test set.
from transformers.utils.notebook import NotebookProgressCallback
import datasets as _ds

dpo_test = load_dataset(dataset_id, data_files="dpo_test.jsonl", split="train")
dpo_test = dpo_test.map(normalize_dpo)
dpo_test = dpo_test.filter(lambda x: x["prompt"] is not None)
print(f"DPO test: {len(dpo_test)} rows")

_orig_ds_map = _ds.Dataset.map
def _safe_map(self, *args, **kwargs):
    kwargs["num_proc"] = 1
    return _orig_ds_map(self, *args, **kwargs)
_ds.Dataset.map = _safe_map

try:
    processing_class = dpo_trainer.processing_class if hasattr(dpo_trainer, "processing_class") else dpo_trainer.tokenizer
    dpo_test_prepared = dpo_trainer._prepare_dataset(dpo_test, processing_class, dpo_trainer.args, "test")
finally:
    _ds.Dataset.map = _orig_ds_map

dpo_trainer.remove_callback(NotebookProgressCallback)

dpo_test_results = dpo_trainer.evaluate(eval_dataset=dpo_test_prepared)
print("DPO Test Metrics:", {k: round(v, 4) for k, v in dpo_test_results.items() if isinstance(v, float)})
print(f"DPO Test Loss: {dpo_test_results.get('eval_loss', 'N/A')}")


dpo_test.jsonl:   0%|          | 0.00/1.32M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/664 [00:00<?, ? examples/s]

Filter:   0%|          | 0/664 [00:00<?, ? examples/s]

DPO test: 664 rows


Extracting prompt in test dataset (num_proc=1):   0%|          | 0/664 [00:00<?, ? examples/s]

Applying chat template to test dataset (num_proc=1):   0%|          | 0/664 [00:00<?, ? examples/s]

Tokenizing test dataset (num_proc=1):   0%|          | 0/664 [00:00<?, ? examples/s]

/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and

{'eval_loss': '0.0006332', 'eval_runtime': '481.8', 'eval_samples_per_second': '1.378', 'eval_steps_per_second': '1.378', 'eval_rewards/chosen': '11.49', 'eval_rewards/rejected': '-6.4', 'eval_rewards/accuracies': '1', 'eval_rewards/margins': '17.89', 'eval_logps/chosen': '-53.73', 'eval_logps/rejected': '-195.1', 'eval_logits/chosen': '0.7402', 'eval_logits/rejected': '0.3064', 'epoch': '1'}
DPO Test Metrics: {'eval_loss': 0.0006, 'eval_runtime': 481.7768, 'eval_samples_per_second': 1.378, 'eval_steps_per_second': 1.378, 'eval_rewards/chosen': 11.4892, 'eval_rewards/rejected': -6.4002, 'eval_rewards/accuracies': 1.0, 'eval_rewards/margins': 17.8894, 'eval_logps/chosen': -53.7299, 'eval_logps/rejected': -195.0949, 'eval_logits/chosen': 0.7402, 'eval_logits/rejected': 0.3064, 'epoch': 1.0}
DPO Test Loss: 0.0006331990007311106


In [13]:
# 8. Save LoRA adapter and back it up to GCS.
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

print(f"Saving adapter locally -> {LOCAL_ADAPTER_DIR} ...")
model.save_pretrained(LOCAL_ADAPTER_DIR, save_method="lora")
tokenizer.save_pretrained(LOCAL_ADAPTER_DIR)
print("Local adapter saved")

print(f"Syncing adapter to GCS -> {GCS_ADAPTER_DIR} ...")
ret = subprocess.run(
    ["gsutil", "-m", "rsync", "-r", LOCAL_ADAPTER_DIR, GCS_ADAPTER_DIR],
    capture_output=True, text=True,
)
if ret.returncode == 0:
    print(f"GCS adapter backup OK: {GCS_ADAPTER_DIR}")
else:
    print(f"gsutil failed: {ret.stderr[:300]}")
    print("Adapter remains available locally for manual upload.")


Saving adapter locally -> empathAI-lora-adapter-l4-v2 ...


Unsloth: Restored added_tokens_decoder metadata in empathAI-lora-adapter-l4-v2/tokenizer_config.json.


Local adapter saved
Syncing adapter to GCS -> gs://empathai-llama/empathAI-finetune/adapter-l4-v2 ...
GCS adapter backup OK: gs://empathai-llama/empathAI-finetune/adapter-l4-v2


In [ ]:
# 8b. Export and push Hugging Face release branches.
# v2-bf16 and v2-gguf are exported first to preserve maximum quality.
# main is exported last as a forced merged 4-bit inference-ready release.
import gc
import os
import shutil
from pathlib import Path

import torch
from huggingface_hub import HfApi, create_repo, upload_folder
from unsloth import FastLanguageModel

hf_repo_name = os.environ.get("HF_MODEL_REPO", "thanhhoangnvbg/empathAI-llama3.1-8b")
hf_main_branch = os.environ.get("HF_MAIN_BRANCH", "main")
hf_bf16_branch = os.environ.get("HF_BF16_BRANCH", "v2-bf16")
hf_gguf_branch = os.environ.get("HF_GGUF_BRANCH", "v2-gguf")
gguf_quants = [
    q.strip()
    for q in os.environ.get("GGUF_QUANTS", "q4_k_m,q5_k_m").split(",")
    if q.strip()
]
export_root = Path(os.environ.get("HF_EXPORT_DIR", f"empathAI-hf-export-{RUN_NAME}"))
max_memory_usage = float(os.environ.get("EXPORT_MAXIMUM_MEMORY_USAGE", "0.60"))

push_main_4bit = os.environ.get("PUSH_MAIN_4BIT", "1") != "0"
push_bf16 = os.environ.get("PUSH_BF16", "1") != "0"
push_gguf = os.environ.get("PUSH_GGUF", "1") != "0"

if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN":
    raise RuntimeError("Set HF_TOKEN or HUGGINGFACE_HUB_TOKEN before pushing to Hugging Face.")
if not any([push_main_4bit, push_bf16, push_gguf]):
    raise RuntimeError("All push targets are disabled. Enable at least one of PUSH_MAIN_4BIT/PUSH_BF16/PUSH_GGUF.")

api = HfApi(token=HF_TOKEN)
create_repo(hf_repo_name, repo_type="model", token=HF_TOKEN, exist_ok=True)

def _cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _branch_commits():
    refs = api.list_repo_refs(repo_id=hf_repo_name, repo_type="model")
    return {branch.name: branch.target_commit for branch in refs.branches}


def _ensure_branch(branch):
    if branch == hf_main_branch:
        return
    if branch in _branch_commits():
        return
    api.create_branch(
        repo_id=hf_repo_name,
        repo_type="model",
        branch=branch,
        revision=hf_main_branch,
        token=HF_TOKEN,
        exist_ok=True,
    )


def _reset_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def _write_model_card(folder, branch, variant):
    readme_lines = [
        "---",
        "library_name: transformers",
        "license: mit",
        "base_model: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
        "tags:",
        "- llama",
        "- unsloth",
        "- vietnamese",
        "- customer-service",
        "---",
        "",
        "# EmpathAI Llama 3.1 8B",
        "",
        f"This branch contains the {variant} export of EmpathAI.",
        "",
        "## Branch layout",
        "",
        "| Branch | Contents |",
        "| --- | --- |",
        "| main | latest merged 4-bit inference-ready release |",
        "| v2-bf16 | EmpathAI v2 merged 16-bit weights |",
        "| v2-gguf | EmpathAI v2 GGUF exports: Q4_K_M and Q5_K_M |",
        "| v1-bf16 | archived EmpathAI v1 merged 16-bit weights |",
        "| v1-gguf | archived EmpathAI v1 GGUF exports |",
    ]
    (folder / "README.md").write_text("\n".join(readme_lines) + "\n", encoding="utf-8")


def _upload_clean_branch(folder, branch, commit_message):
    _ensure_branch(branch)
    commit = upload_folder(
        repo_id=hf_repo_name,
        repo_type="model",
        folder_path=str(folder),
        revision=branch,
        token=HF_TOKEN,
        commit_message=commit_message,
        delete_patterns=["*"],
    )
    print(f"Uploaded {folder} -> https://huggingface.co/{hf_repo_name}/tree/{branch}")
    return commit


def _export_and_upload_merged(save_method, folder, branch, variant):
    folder = _reset_dir(folder)
    print(f"Saving {save_method} -> {folder}")
    model.save_pretrained_merged(
        str(folder),
        tokenizer,
        save_method=save_method,
        maximum_memory_usage=max_memory_usage,
    )
    _write_model_card(folder, branch, variant)
    _cuda_cleanup()
    return _upload_clean_branch(folder, branch, f"v2: upload {variant}")


def _export_and_upload_gguf(folder, branch):
    folder = _reset_dir(folder)
    print(f"Saving GGUF {gguf_quants} -> {folder}")
    model.save_pretrained_gguf(
        str(folder),
        tokenizer,
        quantization_method=gguf_quants,
        maximum_memory_usage=max_memory_usage,
    )
    _write_model_card(folder, branch, "GGUF Q4_K_M and Q5_K_M")
    _cuda_cleanup()
    return _upload_clean_branch(folder, branch, f"v2: upload GGUF {', '.join(gguf_quants)}")


FastLanguageModel.for_inference(model)
_cuda_cleanup()
if torch.cuda.is_available():
    print("Free VRAM before export:", round(torch.cuda.mem_get_info()[0] / 1e9, 2), "GB")

print(f"Hub repo: https://huggingface.co/{hf_repo_name}")
print(f"Export root: {export_root}")

# Export higher-quality formats first. Unsloth intentionally blocks plain
# merged_4bit before other exports because 4-bit merging is a final lossy step.
if push_bf16:
    _export_and_upload_merged(
        "merged_16bit",
        export_root / "v2-bf16",
        hf_bf16_branch,
        "merged 16-bit BF16 release",
    )

if push_gguf:
    _export_and_upload_gguf(export_root / "v2-gguf", hf_gguf_branch)

if push_main_4bit:
    _export_and_upload_merged(
        "merged_4bit_forced",
        export_root / "main-4bit",
        hf_main_branch,
        "merged 4-bit inference build",
    )

print("Done. Branch commits:")
commits = _branch_commits()
for branch in [hf_main_branch, hf_bf16_branch, hf_gguf_branch]:
    commit = commits.get(branch, "")
    print(f"- {branch}: {commit[:12]} https://huggingface.co/{hf_repo_name}/tree/{branch}")


In [17]:
# 9. Optional local smoke test for the in-memory trained model.
# Set RUN_LOCAL_SMOKE=1 if you want this to run after training.
import os

if os.environ.get("RUN_LOCAL_SMOKE") != "1":
    print("Skipping local smoke test. Set RUN_LOCAL_SMOKE=1 to generate one sample response.")
else:
    from unsloth import FastLanguageModel

    FastLanguageModel.for_inference(model)

    messages = [
        {"role": "system", "content": "Bạn là EmpathAI, trợ lý chăm sóc khách hàng e-commerce tiếng Việt. Trả lời ngắn gọn, bình tĩnh, không tự bịa trạng thái đơn hàng hoặc mức bồi thường."},
        {"role": "user", "content": "Shop giao hàng quá dở, tôi chờ 3 ngày rồi mà vẫn chưa nhận được."},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=180,
        temperature=0.4,
        top_p=0.9,
        do_sample=True,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(response.strip())


Skipping local smoke test. Set RUN_LOCAL_SMOKE=1 to generate one sample response.


In [ ]:
# 9b. Optional: verify the pushed Hub model in a fresh runtime.
# Keep this disabled by default so Run All does not reload another 8B model into L4 VRAM.
import os

if os.environ.get("TEST_PUSHED_MODEL") != "1":
    print("Skipping pushed-model reload. Set TEST_PUSHED_MODEL=1 in a fresh runtime to verify the Hub repo.")
else:
    import gc
    import torch
    from unsloth import FastLanguageModel

    if "model" in globals():
        del model
    gc.collect()
    torch.cuda.empty_cache()

    model_name = os.environ.get("HF_MODEL_REPO", "thanhhoangnvbg/empathAI-llama3.1-8b")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=2048,
        load_in_4bit=True,
        token=HF_TOKEN or None,
    )
    FastLanguageModel.for_inference(model)

    messages = [
        {"role": "system", "content": "Bạn là EmpathAI, trợ lý chăm sóc khách hàng e-commerce tiếng Việt. Trả lời ngắn gọn, bình tĩnh, không tự bịa trạng thái đơn hàng hoặc mức bồi thường."},
        {"role": "user", "content": "Tôi rất thất vọng vì đơn hàng bị giao chậm 3 ngày rồi!"},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=180,
        use_cache=True,
        temperature=0.4,
        top_p=0.9,
    )

    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(response.strip())
